# FewShot Combined CONCH v2 — Pathologist-Reviewed + CRC Top-Up Patches

## Methodological Assessment

### 1. Pipeline Structure
load model → extract features → combine datasets → **20-per-class safety cap** → **80/20 train/test split** → train classifiers → evaluate on test set → save classifier → run inference.

### 2. Training Set Composition
- **Pathologist (Step 2a):** 101 expert-reviewed patches across 13 classes
- **CRC top-up (Step 2b):** 79 CRC100K patches, bringing 6 classes up to 20 samples each
- **Combined total:** 180 patches, 13 classes (all from `fewshot_combined_to20.xlsx`)
- **20-per-class safety cap:** already satisfied by design; cap guards against any accidental overlap
- **Train / Test split (80/20 stratified):** ~144 train / ~36 test

### 3. BACK Removal
- **Training:** no BACK patches exist in `fewshot_combined_to20.xlsx` — not needed.
- **Inference (Step 7):** patches loaded from `tcga_clean_no_back.csv` (pre-filtered by BACK SVM).

### 4. Data Sparsity
- **PLC: 2 patches** → 1 train / 1 test. Treat PLC metrics as qualitative only.
- **SIG: 6, MES: 8** — sparse but usable for few-shot.
- **LYA: 12, ADE/CAR: 11, PDC: 10** — reasonable for few-shot.

### 5. Design Choices
- `class_weight='balanced'` in LogReg handles class imbalance.
- All outputs go to `Results/Few_Shot_Evaluation/Combined/` — original files never overwritten.

In [1]:
import os
import json
import numpy as np
import pandas as pd
import torch
import joblib
import matplotlib

matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay
)
# from sklearn.model_selection import LeaveOneOut, cross_val_predict

import sys
sys.argv = ['']

from conch.open_clip_custom import create_model_from_pretrained
from conch.downstream.zeroshot_path import zero_shot_classifier

print('Imports OK')

Imports OK


c:\Users\datainsight\anaconda3\envs\exaonepath\lib\site-packages\timm\models\layers\__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:
# ── Paths (forward slashes work on Windows) ──────────────────────────────────
CONCH_CKPT    = 'checkpoints/conch/pytorch_model.bin'
PROMPT_CRC    = 'config_files/CRC100K_prompts_all_per_class.json'
PROMPT_CUSTOM = 'config_files/Custom_prompts_all_per_class.json'

# Pathologist + top-up feature caches (from fewshot_combined_to20.xlsx)
PATH_CACHE  = 'Results/Few_shot_features/conch_pathologist_features.pt'
TOPUP_CACHE = 'Results/Few_shot_features/conch_crc_topup_features.pt'

# Full CRC-100K feature cache — used only to sample 20 LYM patches
CRC_CACHE = 'Results/Few_shot_features/conch_train_features.pt'

# Excel with all labelled rows (101 pathologist + 99 top-up incl. LYM)
EXCEL_FILE = 'fewshot_combined_to20.xlsx'

# CRC-100K image roots — pathologist patches are CLASS-XXXXXXXX.tif
CRC_DIR     = 'D:/Aamir Gulzar/dataset/CRC100K/NCT-CRC-HE-100K-NONORM'
CRC_VAL_DIR = 'D:/Aamir Gulzar/dataset/CRC100K/CRC-VAL-HE-7K'

# TCGA patch root (inference only)
PATCH_DIR  = 'D:/Aamir Gulzar/KSA_project2/dataset/patch_data'

# Pre-extracted CONCH v1 five-crop features for TCGA patches (Step 7)
# Each .pt file: [5, 512] tensor — mean-pooled → L2-normalised at inference
# Use v1 (512-dim), NOT v1.5 (768-dim) — classifier was trained on 512-dim features
FEAT_DIR_V1 = 'D:/Aamir Gulzar/KSA_project2/dataset/Features/Conch_v1_fivecrop'

# Pre-filtered inference patch list (BACK removed by SVM — TCGA only)
TCGA_CLEAN_CSV = 'D:/Aamir Gulzar/KSA_project2/dataset/tcga_clean_no_back.csv'

# ── Output paths (Combined subfolder — originals never overwritten) ───────────
COMBINED_RESULTS_DIR = 'Results/Few_Shot_Evaluation/Combined'
COMBINED_CLF_OUT     = 'Results/Few_Shot_Evaluation/Combined/combined_fewshot_logistic_regression.pkl'
ZS_WEIGHTS_OUT       = 'Results/combined_zeroshot_weights_v2.pt'
PROTO_OUT            = 'Results/few_shot_combined_prototypes_v2.pt'
CM_OUT               = 'Results/Few_Shot_Evaluation/Combined/confusion_matrix_combined_fewshot_v2.png'
TCGA_CSV_OUT         = 'Results/classification_results_combined_fewshot_v2.csv'
TCGA_CSV_ORIG        = 'Results/classification_results_tcga_fewshot.csv'

FEW_SHOT_CAP = 20   # max samples per class (top-up already ensures ≤20; cap is a safety guard)
RANDOM_STATE = 42

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

os.makedirs('Results', exist_ok=True)
os.makedirs('Results/Few_shot_features', exist_ok=True)
os.makedirs(COMBINED_RESULTS_DIR, exist_ok=True)
print('Output directories ready.')


Device: cuda:0
Output directories ready.


In [3]:
# ── Load CONCH model (shared across all steps) ───────────────────────────────
print('Loading CONCH model...')
model, preprocess = create_model_from_pretrained(
    model_cfg='conch_ViT-B-16',
    checkpoint_path=CONCH_CKPT,
    device=DEVICE,
    force_image_size=224,
)
_ = model.eval()
print('CONCH model loaded.')

Loading CONCH model...
CONCH model loaded.


## Step 2 — Load Pathologist + CRC Top-Up Features


In [4]:
print('=' * 60)
print('STEP 2 — Loading pathologist + CRC top-up features')
print('=' * 60)

# --- 2A: Original 101 pathologist patches ------------------------------------
if not os.path.exists(PATH_CACHE):
    print(f'  PATH_CACHE not found — extracting from non-TopUp rows of {EXCEL_FILE} ...')
    df_all = pd.read_excel(EXCEL_FILE, usecols=['Filename', 'Primary Label', 'Source Sheet'])

    # Pathologist rows = everything that is NOT the CRC TopUp sheet
    df_path_orig = df_all[df_all['Source Sheet'].str.strip() != 'CRC100K TopUp'].copy()
    df_path_orig = df_path_orig.dropna(subset=['Filename', 'Primary Label']).reset_index(drop=True)
    df_path_orig['Primary Label'] = df_path_orig['Primary Label'].str.strip()
    df_path_orig['Filename']      = df_path_orig['Filename'].astype(str).str.strip()

    # All pathologist patches are CRC-100K images: CLASS-XXXXXXXX.tif
    # Primary root is NCT-CRC-HE-100K-NONORM; TCGA-prefixed files fall in CRC-VAL-HE-7K.
    def make_crc_path(filename):
        crc_class   = filename.split('-')[0]   # e.g. 'LYM' from 'LYM-RLPSHGFW.tif'
        primary     = os.path.join(CRC_DIR,     crc_class, filename)
        fallback    = os.path.join(CRC_VAL_DIR, crc_class, filename)
        return primary if os.path.exists(primary) else fallback

    df_path_orig['img_path'] = df_path_orig['Filename'].apply(make_crc_path)

    found_mask = df_path_orig['img_path'].apply(os.path.exists)
    missing_p  = df_path_orig[~found_mask]
    df_path_orig = df_path_orig[found_mask].reset_index(drop=True)

    if len(missing_p):
        print(f'  [WARN] {len(missing_p)} files not found in CRC_DIR or CRC_VAL_DIR:')
        for p in missing_p['img_path'].tolist()[:5]:
            print(f'    {p}')
    print(f'  Extracting features for {len(df_path_orig)} pathologist patches...')

    feats_p, lbls_p = [], []
    with torch.inference_mode():
        for _, row in tqdm(df_path_orig.iterrows(), total=len(df_path_orig),
                           desc='Pathologist patches'):
            try:
                img  = Image.open(row['img_path']).convert('RGB')
                t    = preprocess(img).unsqueeze(0).to(DEVICE)
                feat = model.encode_image(t)
                feat = feat / feat.norm(dim=-1, keepdim=True)
                feats_p.append(feat.cpu().squeeze(0))
                lbls_p.append(row['Primary Label'])
            except Exception as e:
                print(f'  [WARN] {row["img_path"]}: {e}')

    if not feats_p:
        raise RuntimeError('No pathologist features extracted — check CRC_DIR/CRC_VAL_DIR and Filename column.')

    X_path_t = torch.stack(feats_p)
    y_path   = np.array(lbls_p)
    torch.save({'features': X_path_t, 'labels': y_path}, PATH_CACHE)
    print(f'  Saved {X_path_t.shape[0]} pathologist features -> {PATH_CACHE}')
else:
    cache_p  = torch.load(PATH_CACHE, weights_only=False)
    X_path_t = cache_p['features']
    y_path   = cache_p['labels']
    print(f'  Pathologist cache loaded: {X_path_t.shape[0]} features  [{PATH_CACHE}]')

# --- 2B: 79 CRC top-up patches (from FewShot_TopUp_To20.ipynb) --------------
if not os.path.exists(TOPUP_CACHE):
    raise FileNotFoundError(
        f'TOPUP_CACHE not found: {TOPUP_CACHE}\n'
        'Run FewShot_TopUp_To20.ipynb first to generate it.'
    )
cache_tu = torch.load(TOPUP_CACHE, weights_only=False)
X_topup  = cache_tu['features']
y_topup  = cache_tu['labels']
print(f'  TopUp cache loaded:       {X_topup.shape[0]} features  [{TOPUP_CACHE}]')

path_classes  = sorted(set(y_path.tolist()))
topup_classes = sorted(set(y_topup.tolist()))
print(f'\n  Pathologist classes ({len(path_classes)}): {path_classes}')
print(f'  TopUp classes       ({len(topup_classes)}): {topup_classes}')

# Row-count sanity check against Excel
df_excel = pd.read_excel(EXCEL_FILE, usecols=['Primary Label'])
df_excel = df_excel.dropna(subset=['Primary Label'])
total_excel  = len(df_excel)
total_caches = len(y_path) + len(y_topup)
status = 'OK' if total_excel == total_caches else 'MISMATCH — recheck caches'
print(f'\n  Excel rows: {total_excel} | Cached total: {total_caches} | {status}')
print('  Label distribution from Excel:')
print(df_excel['Primary Label'].value_counts().to_string())


STEP 2 — Loading pathologist + CRC top-up features
  Pathologist cache loaded: 101 features  [Results/Few_shot_features/conch_pathologist_features.pt]
  TopUp cache loaded:       99 features  [Results/Few_shot_features/conch_crc_topup_features.pt]

  Pathologist classes (13): ['ADE', 'ADI', 'CAR', 'DEB', 'LYA', 'MES', 'MUC', 'MUS', 'NORM', 'PDC', 'PLC', 'SIG', 'STR']
  TopUp classes       (7): ['ADI', 'DEB', 'LYM', 'MUC', 'MUS', 'NORM', 'STR']

  Excel rows: 200 | Cached total: 200 | OK
  Label distribution from Excel:
Primary Label
MUS     20
STR     20
ADI     20
DEB     20
MUC     20
NORM    20
LYM     20
LYA     12
ADE     11
CAR     11
PDC     10
MES      8
SIG      6
PLC      2


## Step 3 — Build Unified Label Space and Prompt Dict

In [5]:
print('=' * 60)
print('STEP 3 — Unified label space')
print('=' * 60)

with open(PROMPT_CRC) as f:
    crc_data = json.load(f)['0']
with open(PROMPT_CUSTOM) as f:
    custom_data = json.load(f)['0']

# Merge: CRC100K is base; Custom adds new classes only
unified_classnames = dict(crc_data['classnames'])
for cls, syns in custom_data['classnames'].items():
    if cls not in unified_classnames:
        unified_classnames[cls] = syns

templates = crc_data['templates']

# Stable order: CRC classes first, then new ones
unified_classes = list(crc_data['classnames'].keys())
for cls in custom_data['classnames'].keys():
    if cls not in unified_classes:
        unified_classes.append(cls)

# Training labels come from pathologist + top-up only; BACK must never appear
all_train_labels = set(y_path.tolist()) | set(y_topup.tolist())
all_train_labels.discard('BACK')

# Drop any class with no training samples
for cls in list(unified_classes):
    if cls not in all_train_labels:
        unified_classes.remove(cls)
        unified_classnames.pop(cls, None)
        print(f'  [INFO] Dropped "{cls}" — no training samples.')

print(f'  Unified classes ({len(unified_classes)}): {unified_classes}')
print('  Synonyms per class:')
for cls in unified_classes:
    print(f'    {cls:6s}: {unified_classnames[cls]}')


STEP 3 — Unified label space
  [INFO] Dropped "BACK" — no training samples.
  [INFO] Dropped "TUM" — no training samples.
  Unified classes (14): ['ADI', 'DEB', 'LYM', 'MUC', 'MUS', 'NORM', 'STR', 'PLC', 'LYA', 'ADE', 'MES', 'CAR', 'PDC', 'SIG']
  Synonyms per class:
    ADI   : ['adipose', 'adipose tissue', 'adipocytes', 'fat', 'fat cells']
    DEB   : ['debris', 'necrosis', 'necrotic tissue', 'necrotic debris', 'tissue necrosis']
    LYM   : ['lymphocytes', 'immune cells', 'lymphoid cells', 'white blood cells', 'lymphoid infiltrate']
    MUC   : ['mucus', 'mucin', 'mucus pool', 'mucin pool']
    MUS   : ['smooth muscle', 'smooth muscle tissue', 'muscle', 'muscularis propria', 'muscularis mucosa']
    NORM  : ['normal mucosa', 'normal colon mucosa', 'benign mucosa', 'uninvolved mucosa']
    STR   : ['tumor-associated stroma', 'cancer-associated stroma', 'stromal cells', 'stromal tissue', 'stroma']
    PLC   : ['plasma cells', 'antibody-producing cells', 'plasmocytes', 'B-cell derived 

## Step 4 — Combine Feature Matrices

In [6]:
print('=' * 60)
print('STEP 4 — Combine pathologist + top-up, cap at 20/class, train/test split')
print('=' * 60)

from sklearn.model_selection import train_test_split

# ── 4A: Combine pathologist (101) + top-up (79) = 180 patches ────────────────
X_combined = np.vstack([X_path_t.numpy(), X_topup.numpy()])
y_combined = np.concatenate([y_path, y_topup])

print(f'  Combined: {X_combined.shape[0]} patches, {len(np.unique(y_combined))} classes')
print('  Class counts:')
vals, cnts = np.unique(y_combined, return_counts=True)
for v, c in sorted(zip(vals, cnts)):
    src = []
    if v in path_classes:  src.append('Path')
    if v in topup_classes: src.append('TopUp')
    flag = ' ← will be capped' if c > FEW_SHOT_CAP else ''
    print(f'    {v:6s}: {c:3d}  ({" + ".join(src)}){flag}')

# ── 4B: Safety cap at FEW_SHOT_CAP (=20) per class ───────────────────────────
# Top-up already ensures ≤20 per class; this guards against any unexpected overlap
rng = np.random.default_rng(RANDOM_STATE)
cap_idx = []
for cls in sorted(np.unique(y_combined)):
    idx = np.where(y_combined == cls)[0].tolist()
    rng.shuffle(idx)
    cap_idx.extend(idx[:FEW_SHOT_CAP])
cap_idx = sorted(cap_idx)

X_capped = X_combined[cap_idx]
y_capped = y_combined[cap_idx]

print(f'\n  After {FEW_SHOT_CAP}-per-class cap: {X_capped.shape[0]} patches')

# ── 4C: 80/20 stratified train/test split ────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_capped, y_capped,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_capped,
)

print(f'  Train: {X_train.shape[0]} patches  |  Test: {X_test.shape[0]} patches')
print('  Train class distribution:')
vals_tr, cnts_tr = np.unique(y_train, return_counts=True)
for v, c in sorted(zip(vals_tr, cnts_tr)):
    print(f'    {v:6s}: {c}')
print('  Test class distribution:')
vals_te, cnts_te = np.unique(y_test, return_counts=True)
for v, c in sorted(zip(vals_te, cnts_te)):
    print(f'    {v:6s}: {c}')


STEP 4 — Combine pathologist + top-up, cap at 20/class, train/test split
  Combined: 200 patches, 14 classes
  Class counts:
    ADE   :  11  (Path)
    ADI   :  20  (Path + TopUp)
    CAR   :  11  (Path)
    DEB   :  20  (Path + TopUp)
    LYA   :  12  (Path)
    LYM   :  20  (TopUp)
    MES   :   8  (Path)
    MUC   :  20  (Path + TopUp)
    MUS   :  20  (Path + TopUp)
    NORM  :  20  (Path + TopUp)
    PDC   :  10  (Path)
    PLC   :   2  (Path)
    SIG   :   6  (Path)
    STR   :  20  (Path + TopUp)

  After 20-per-class cap: 200 patches
  Train: 160 patches  |  Test: 40 patches
  Train class distribution:
    ADE   : 9
    ADI   : 16
    CAR   : 9
    DEB   : 16
    LYA   : 9
    LYM   : 16
    MES   : 6
    MUC   : 16
    MUS   : 16
    NORM  : 16
    PDC   : 8
    PLC   : 2
    SIG   : 5
    STR   : 16
  Test class distribution:
    ADE   : 2
    ADI   : 4
    CAR   : 2
    DEB   : 4
    LYA   : 3
    LYM   : 4
    MES   : 2
    MUC   : 4
    MUS   : 4
    NORM  : 4
    PDC   :

## Step 5 — Train All Three Classifiers on Combined Set

Self-evaluation on the full training set is shown for reference. 
For unbiased estimates, see Step 6 (LOO CV).

In [7]:
# ── 5A: Zero-Shot ─────────────────────────────────────────────────────────────
print('=' * 60)
print('STEP 5A — Zero-Shot  (text prompts, no training data used)')
print('=' * 60)

classnames_text = [unified_classnames[c] for c in unified_classes]

# Invalidate stale cache if the class count has changed
if os.path.exists(ZS_WEIGHTS_OUT):
    _peek = torch.load(ZS_WEIGHTS_OUT, map_location='cpu', weights_only=False)
    if _peek.shape[1] != len(unified_classes):
        print(f'  [INFO] Stale ZS cache ({_peek.shape[1]} classes ≠ {len(unified_classes)} current) — regenerating.')
        del _peek
        os.remove(ZS_WEIGHTS_OUT)
    else:
        del _peek

if os.path.exists(ZS_WEIGHTS_OUT):
    print(f'  Loading cached ZS weights from {ZS_WEIGHTS_OUT} ...')
    zs_weights = torch.load(ZS_WEIGHTS_OUT, map_location=DEVICE, weights_only=False)
else:
    print('  Generating zero-shot weights from text prompts...')
    zs_weights = zero_shot_classifier(model, classnames_text, templates, device=DEVICE)
    torch.save(zs_weights, ZS_WEIGHTS_OUT)
    print(f'  Saved -> {ZS_WEIGHTS_OUT}')

print(f'  Weight matrix: {zs_weights.shape}  ({len(unified_classes)} classes)')

# Evaluate on TEST set
X_test_t  = torch.tensor(X_test, dtype=torch.float32).to(DEVICE)
zs_logits = (100.0 * X_test_t @ zs_weights).cpu()
zs_preds  = [unified_classes[i] for i in zs_logits.argmax(dim=-1)]

zs_acc  = accuracy_score(y_test, zs_preds)
zs_bacc = balanced_accuracy_score(y_test, zs_preds)
print(f'\n  Zero-Shot  |  Acc: {zs_acc:.4f}  |  Balanced Acc: {zs_bacc:.4f}')
print('  (evaluated on test set — ZS uses no training data)')

STEP 5A — Zero-Shot  (text prompts, no training data used)
  Loading cached ZS weights from Results/combined_zeroshot_weights_v2.pt ...
  Weight matrix: torch.Size([512, 14])  (14 classes)

  Zero-Shot  |  Acc: 0.7500  |  Balanced Acc: 0.7308
  (evaluated on test set — ZS uses no training data)


In [8]:
# ── 5B: Nearest Centroid / ProtoNet ───────────────────────────────────────────
print('=' * 60)
print('STEP 5B — Nearest Centroid / ProtoNet')
print('=' * 60)

def compute_prototypes(X, y, classes):
    protos = []
    for cls in classes:
        mask = y == cls
        if not np.any(mask):
            print(f'  [WARN] No training samples for "{cls}" — zero prototype.')
            protos.append(torch.zeros(X.shape[1]))
            continue
        v = X[mask].mean(dim=0)
        protos.append(v / v.norm())
    return torch.stack(protos)

# Build prototypes from TRAINING set only
train_classes_present = [c for c in unified_classes if c in np.unique(y_train)]
X_train_t  = torch.tensor(X_train, dtype=torch.float32)
prototypes = compute_prototypes(X_train_t, y_train, train_classes_present)

# Evaluate on TEST set
X_test_t     = torch.tensor(X_test, dtype=torch.float32)
proto_logits = (100.0 * X_test_t.to(DEVICE) @ prototypes.to(DEVICE).T).cpu()
proto_preds  = [train_classes_present[i] for i in proto_logits.argmax(dim=-1)]

proto_acc  = accuracy_score(y_test, proto_preds)
proto_bacc = balanced_accuracy_score(y_test, proto_preds)
print(f'  ProtoNet  |  Acc: {proto_acc:.4f}  |  Balanced Acc: {proto_bacc:.4f}')
print(f'  (trained on {len(y_train)} samples, evaluated on {len(y_test)} test samples)')

torch.save({'prototypes': prototypes.cpu(), 'classes': train_classes_present}, PROTO_OUT)
print(f'  Prototypes saved -> {PROTO_OUT}  ({len(train_classes_present)} classes)')


STEP 5B — Nearest Centroid / ProtoNet
  ProtoNet  |  Acc: 0.8000  |  Balanced Acc: 0.7308
  (trained on 160 samples, evaluated on 40 test samples)
  Prototypes saved -> Results/few_shot_combined_prototypes_v2.pt  (14 classes)


In [9]:
# ── 5C: Logistic Regression (Linear Probe) ────────────────────────────────────
print('=' * 60)
print('STEP 5C — Logistic Regression (Linear Probe)')
print('=' * 60)

clf_combined = LogisticRegression(
    max_iter=1000, C=1.0, class_weight='balanced', random_state=RANDOM_STATE
)

# Train on TRAINING set only — no leakage
clf_combined.fit(X_train, y_train)
print(f'  Trained on {len(y_train)} samples ({len(np.unique(y_train))} classes)')

# Evaluate on TEST set
lr_preds = clf_combined.predict(X_test)
lr_acc   = accuracy_score(y_test, lr_preds)
lr_bacc  = balanced_accuracy_score(y_test, lr_preds)
print(f'  LogReg  |  Acc: {lr_acc:.4f}  |  Balanced Acc: {lr_bacc:.4f}')
print(f'  (evaluated on {len(y_test)} held-out test samples)')

# Save classifier
joblib.dump(clf_combined, COMBINED_CLF_OUT)
print(f'  Saved -> {COMBINED_CLF_OUT}')
print('  Original few_shot_logistic_regression.pkl is NOT modified.')


STEP 5C — Logistic Regression (Linear Probe)
  Trained on 160 samples (14 classes)
  LogReg  |  Acc: 0.7750  |  Balanced Acc: 0.7115
  (evaluated on 40 held-out test samples)
  Saved -> Results/Few_Shot_Evaluation/Combined/combined_fewshot_logistic_regression.pkl
  Original few_shot_logistic_regression.pkl is NOT modified.


## Step 6 — Full Evaluation on Test Set + Summary

Mirrors the original `Few_Shot_Label_Matching_Conch.ipynb` evaluation section:
classification report, confusion matrix, and a side-by-side summary table for all three classifiers.

In [10]:
print('=' * 60)
print('STEP 6 — Full evaluation on test set')
print('=' * 60)

test_labels = sorted(np.unique(y_test).tolist())

all_results = []

# ── Helper ────────────────────────────────────────────────────────────────────
def evaluate_classifier(y_true, y_pred, labels, title, results_dir):
    acc  = accuracy_score(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred)
    print(f'\n  ===== {title.upper()} =====')
    print(f'  Accuracy:          {acc:.4f}')
    print(f'  Balanced Accuracy: {bacc:.4f}')
    report = classification_report(y_true, y_pred, labels=labels, zero_division=0)
    print(report)

    cm  = confusion_matrix(y_true, y_pred, labels=labels)
    fig, ax = plt.subplots(figsize=(max(10, len(labels) * 0.9), max(8, len(labels) * 0.7)))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    disp.plot(ax=ax, cmap='Blues', xticks_rotation=45, colorbar=True)
    ax.set_title(f'Confusion Matrix — {title}', fontsize=12, fontweight='bold')
    plt.tight_layout()
    cm_path = os.path.join(results_dir, f'{title.replace(" ", "_").lower()}_cm.png')
    plt.savefig(cm_path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f'  Confusion matrix saved -> {cm_path}')
    return {'title': title, 'accuracy': acc, 'balanced_accuracy': bacc}

# ── Zero-Shot ─────────────────────────────────────────────────────────────────
res = evaluate_classifier(y_test, zs_preds, test_labels,
                           'Zero-Shot', COMBINED_RESULTS_DIR)
all_results.append(res)

# ── ProtoNet ──────────────────────────────────────────────────────────────────
res = evaluate_classifier(y_test, proto_preds, test_labels,
                           f'Few-Shot Nearest Centroid ({len(y_train)} train)',
                           COMBINED_RESULTS_DIR)
all_results.append(res)

# ── LogReg ────────────────────────────────────────────────────────────────────
res = evaluate_classifier(y_test, lr_preds, test_labels,
                           f'Few-Shot Logistic Regression ({len(y_train)} train)',
                           COMBINED_RESULTS_DIR)
all_results.append(res)

# ── Ensemble (ZS + ProtoNet, α=0.5) ──────────────────────────────────────────
import torch.nn.functional as F
zs_prob  = F.softmax(zs_logits,   dim=-1)
fs_prob  = F.softmax(proto_logits, dim=-1)

# Align class order: ZS uses unified_classes, ProtoNet uses train_classes_present
zs_cls_idx = {c: i for i, c in enumerate(unified_classes)}
pr_cls_idx = {c: i for i, c in enumerate(train_classes_present)}
n_test     = len(y_test)
all_cls    = sorted(set(unified_classes) | set(train_classes_present))
zs_full = torch.zeros(n_test, len(all_cls))
pr_full = torch.zeros(n_test, len(all_cls))
for i, cls in enumerate(all_cls):
    if cls in zs_cls_idx:
        zs_full[:, i] = zs_prob[:, zs_cls_idx[cls]]
    if cls in pr_cls_idx:
        pr_full[:, i] = fs_prob[:, pr_cls_idx[cls]]

ens_prob  = 0.5 * zs_full + 0.5 * pr_full
ens_preds = [all_cls[i] for i in ens_prob.argmax(dim=-1)]
res = evaluate_classifier(y_test, ens_preds, test_labels,
                           'Ensemble (ZS + ProtoNet)', COMBINED_RESULTS_DIR)
all_results.append(res)

# ── Summary table (same format as original notebook) ─────────────────────────
print('\n' + '=' * 60)
print('SUMMARY')
print('=' * 60)
summary_rows = [{'Classifier': r['title'],
                 'Accuracy': f"{r['accuracy']:.4f}",
                 'Balanced Accuracy': f"{r['balanced_accuracy']:.4f}"}
                for r in all_results]
summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

summary_csv = os.path.join(COMBINED_RESULTS_DIR, 'summary.csv')
summary_df.to_csv(summary_csv, index=False)
print(f'\nSummary saved -> {summary_csv}')

# ── Comparison bar chart ──────────────────────────────────────────────────────
titles = [r['title'] for r in all_results]
accs   = [r['accuracy'] for r in all_results]
baccs  = [r['balanced_accuracy'] for r in all_results]
x = np.arange(len(titles))
w = 0.35
fig, ax = plt.subplots(figsize=(max(10, len(titles) * 2.5), 6))
b1 = ax.bar(x - w/2, accs,  w, label='Accuracy',          color='#4C8BE3')
b2 = ax.bar(x + w/2, baccs, w, label='Balanced Accuracy',  color='#E3844C')
for b in list(b1) + list(b2):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.005,
            f'{b.get_height():.3f}', ha='center', va='bottom', fontsize=8)
ax.set_ylim(0, 1.12)
ax.set_xticks(x)
ax.set_xticklabels(titles, rotation=15, ha='right', fontsize=9)
ax.set_ylabel('Score')
ax.set_title('Classifier Comparison — Combined Few-Shot v2 (15 classes)', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
bar_path = os.path.join(COMBINED_RESULTS_DIR, 'classifier_comparison.png')
plt.savefig(bar_path, dpi=200, bbox_inches='tight')
plt.close()
print(f'Comparison chart saved -> {bar_path}')
print(f'\nAll results in: {COMBINED_RESULTS_DIR}')


STEP 6 — Full evaluation on test set

  ===== ZERO-SHOT =====
  Accuracy:          0.7500
  Balanced Accuracy: 0.7308
              precision    recall  f1-score   support

         ADE       0.50      0.50      0.50         2
         ADI       1.00      1.00      1.00         4
         CAR       0.50      0.50      0.50         2
         DEB       0.80      1.00      0.89         4
         LYA       0.75      1.00      0.86         3
         LYM       1.00      0.75      0.86         4
         MES       0.67      1.00      0.80         2
         MUC       0.75      0.75      0.75         4
         MUS       1.00      1.00      1.00         4
        NORM       1.00      0.50      0.67         4
         PDC       0.00      0.00      0.00         2
         SIG       0.50      1.00      0.67         1
         STR       0.67      0.50      0.57         4

    accuracy                           0.75        40
   macro avg       0.70      0.73      0.70        40
weighted avg    

## Step 7 — Classify All TCGA Patches from Pre-Extracted Features

No image I/O, no CONCH forward pass.  
Each patch already has a `[5, 512]` five-crop feature tensor in `Conch_v1_fivecrop/`.  
Step 7 mean-pools the 5 crops → L2-normalises → runs `clf_inference.predict()`.

Two-pass design:
1. **Pass 1** — load all `.pt` files into a RAM buffer (~2.8 GB for 1.35 M patches)
2. **Pass 2** — one `sklearn.predict` call on the full matrix (single BLAS multiply)

In [12]:
print('=' * 60)
print('STEP 7 -- Classifying TCGA patches from pre-extracted features')
print('=' * 60)

from concurrent.futures import ThreadPoolExecutor

# ── Load classifier and patch list ───────────────────────────────────────────
if not os.path.exists(COMBINED_CLF_OUT):
    raise FileNotFoundError(
        f'Classifier not found: {COMBINED_CLF_OUT}\n'
        'Run Steps 4-5 first to train and save the classifier.'
    )
clf_inference = joblib.load(COMBINED_CLF_OUT)
print(f'  Classifier loaded from {COMBINED_CLF_OUT}')

if not os.path.exists(TCGA_CLEAN_CSV):
    raise FileNotFoundError(f'TCGA clean CSV not found: {TCGA_CLEAN_CSV}')
patch_names = pd.read_csv(TCGA_CLEAN_CSV, usecols=['patch_name'])['patch_name'].tolist()
n_total = len(patch_names)
print(f'  Patches to classify: {n_total:,}')
print(f'  Feature store: {FEAT_DIR_V1}')

# ── Pre-allocate buffers ─────────────────────────────────────────────────────
feat_buf   = np.empty((n_total, 512), dtype=np.float32)
name_list  = [None] * n_total
valid_mask = np.zeros(n_total, dtype=bool)
errors     = 0

# ── Pass 1: parallel .pt loading with ThreadPoolExecutor ─────────────────────
# torch.load releases the GIL during disk I/O, so threads work great here
# and avoid the multiprocessing stalls that DataLoader workers cause on Windows
def load_one(args):
    i, name = args
    stem  = os.path.splitext(name)[0]            # strip .png
    slide = stem.split('_x')[0]                   # e.g. 'TCGA-3L-AA1B_nonMSIH'
    path  = os.path.join(FEAT_DIR_V1, slide, stem + '.pt')
    try:
        feat = torch.load(path, weights_only=True)   # [5, 512]
        feat = feat.mean(dim=0)                       # [512]
        feat = feat / feat.norm().clamp(min=1e-8)     # L2 normalise
        return i, name, feat.numpy(), True
    except Exception:
        return i, name, None, False

print(f'\n  Pass 1 — Loading pre-extracted .pt features  (threads=16)')
with ThreadPoolExecutor(max_workers=16) as ex:
    for i, name, feat, ok in tqdm(
            ex.map(load_one, enumerate(patch_names), chunksize=100),
            total=n_total, desc='Loading features'):
        name_list[i] = name
        if ok:
            feat_buf[i]   = feat
            valid_mask[i] = True
        else:
            errors += 1

name_arr = np.array(name_list)
if errors:
    print(f'  [WARN] {errors:,} patches missing from feature store — skipped.')

# ── Pass 2: classify entire matrix in one shot ───────────────────────────────
print(f'\n  Pass 2 — Classifier on {valid_mask.sum():,} features ...')
feat_valid = feat_buf[valid_mask]
preds      = clf_inference.predict(feat_valid)
probas     = clf_inference.predict_proba(feat_valid).max(axis=1)

# ── Write results ────────────────────────────────────────────────────────────
if os.path.exists(TCGA_CSV_OUT):
    os.remove(TCGA_CSV_OUT)

pd.DataFrame({
    'Filename':        name_arr[valid_mask],
    'Predicted_Label': preds,
    'Confidence':      probas.round(4),
}).to_csv(TCGA_CSV_OUT, index=False)

print(f'  Processed: {valid_mask.sum():,}  |  Skipped (missing .pt): {errors}')
print(f'  Results saved -> {TCGA_CSV_OUT}')

STEP 7 -- Classifying TCGA patches from pre-extracted features
  Classifier loaded from Results/Few_Shot_Evaluation/Combined/combined_fewshot_logistic_regression.pkl
  Patches to classify: 1,355,579
  Feature store: D:/Aamir Gulzar/KSA_project2/dataset/Features/Conch_v1_fivecrop

  Pass 1 — Loading pre-extracted .pt features  (threads=16)


Loading features: 100%|██████████| 1355579/1355579 [23:26<00:00, 963.95it/s] 


  [WARN] 4,403 patches missing from feature store — skipped.

  Pass 2 — Classifier on 1,351,176 features ...
  Processed: 1,351,176  |  Skipped (missing .pt): 4403
  Results saved -> Results/classification_results_combined_fewshot_v2.csv


## Step 8 — Compare Original vs Combined Model Predictions

In [13]:
print('=' * 60)
print('STEP 8 -- Original vs combined model comparison')
print('=' * 60)

if not os.path.exists(TCGA_CSV_ORIG):
    print(f'  [SKIP] Original CSV not found: {TCGA_CSV_ORIG}')
elif not os.path.exists(TCGA_CSV_OUT):
    print('  [SKIP] New CSV not found -- run Step 7 first.')
else:
    df_orig = pd.read_csv(TCGA_CSV_ORIG)
    df_new  = pd.read_csv(TCGA_CSV_OUT)

    # Normalise column names
    # Original: patch_name, label, score  |  New: Filename, Predicted_Label, Confidence
    df_orig = df_orig.rename(columns={'patch_name': 'Filename', 'label': 'label_orig'})
    df_new  = df_new.rename(columns={'Predicted_Label': 'label_new'})

    print(f'  Original CSV: {len(df_orig):,} rows')
    print(f'  New CSV:      {len(df_new):,} rows')

    orig_dist = df_orig['label_orig'].value_counts().rename('Original (9-class)')
    n_cls     = len(unified_classes)
    new_dist  = df_new['label_new'].value_counts().rename(f'Combined ({n_cls}-class)')
    dist_df   = pd.concat([orig_dist, new_dist], axis=1).fillna(0).astype(int)
    dist_df['Delta'] = dist_df[f'Combined ({n_cls}-class)'] - dist_df['Original (9-class)']
    print('  Label distribution comparison:')
    print(dist_df.sort_index().to_string())

    merged  = df_orig[['Filename', 'label_orig']].merge(
                  df_new[['Filename', 'label_new']], on='Filename', how='inner')
    changed = merged[merged['label_orig'] != merged['label_new']]
    pct = 100 * len(changed) / len(merged) if len(merged) else 0.0
    print(f'  Patches in common: {len(merged):,}')
    print(f'  Label changed:     {len(changed):,}  ({pct:.1f}%)')

    if len(changed) > 0:
        print('  Top transitions (original -> new):')
        trans = (changed.groupby(['label_orig', 'label_new'])
                        .size().sort_values(ascending=False))
        print(trans.head(20).to_string())

    orig_9 = {'ADI', 'DEB', 'LYM', 'MUC', 'MUS', 'NORM', 'STR', 'TUM'}
    new_only = set(df_new['label_new'].unique()) - orig_9
    if new_only:
        print(f'  New classes assigned beyond original 9: {sorted(new_only)}')
        for cls in sorted(new_only):
            cnt = (df_new['label_new'] == cls).sum()
            print(f'    {cls:6s}: {cnt:,} patches')
    else:
        print('  No patches assigned outside the original 9 classes.')

STEP 8 -- Original vs combined model comparison
  Original CSV: 1,377,933 rows
  New CSV:      1,351,176 rows
  Label distribution comparison:
      Original (9-class)  Combined (14-class)   Delta
ADI                90845                45662  -45183
BACK               41244                    0  -41244
CAR                    0                  157     157
DEB                79730               356850  277120
LYA                    0                11845   11845
LYM                19016                   12  -19004
MUC                61324                    0  -61324
MUS               152155                 3611 -148544
NORM              102186                    0 -102186
PDC                    0               846556  846556
SIG                    0                21334   21334
STR               203200                65149 -138051
TUM               628233                    0 -628233
  Patches in common: 1,351,176
  Label changed:     1,278,200  (94.6%)
  Top transitions (original ->